**Step 1: Install Required libraries**

In [90]:
!pip install SpeechRecognition
!pip install gtts
!pip install requests
!pip install pygame
!pip install pydub
!pip install newsapi-python
!pip install feedparser

**Step 2: Import Libraries**

In [91]:
import speech_recognition as sr
from gtts import gTTS
from IPython.display import Audio, display
import requests
import datetime
import time
import os

**Step 3: Text-to-Speech Function**

In [134]:
from gtts import gTTS
from IPython.display import Audio, display
import ipywidgets as widgets

audio_output = widgets.Output()

def speak(text, print_text=True):

    if print_text:
        print(f"Assistant: {text}")

    tts = gTTS(text=text, lang='en')

    filename = "response.mp3"
    tts.save(filename)

    with audio_output:
        audio_output.clear_output(wait=True)
        display(Audio(filename, autoplay=True))

speak("Hello, I am your personal assistant")
display(audio_output)

Assistant: Hello, I am your personal assistant


Output()

**Step 4: Speech Recognition Function**

In [94]:
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode

def record_audio():
    js = Javascript("""
    async function recordAudio() {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Start Recording';

      div.appendChild(capture);

      document.body.appendChild(div);

      const stream = await navigator.mediaDevices.getUserMedia({audio:true});

      let recorder = new MediaRecorder(stream);

      let chunks = [];

      capture.onclick = () => {
        recorder.start();

        capture.textContent = 'Recording... Click Again To Stop';

        capture.onclick = () => {
          recorder.stop();

          recorder.ondataavailable = e => {
            chunks.push(e.data);
          }

          recorder.onstop = async () => {
            let blob = new Blob(chunks);

            let reader = new FileReader();

            reader.readAsDataURL(blob);

            reader.onloadend = () => {
              google.colab.kernel.invokeFunction(
                  'notebook.save_audio',
                  [reader.result],
                  {}
              );
            }
          }
        }
      }
    }
    recordAudio();
    """)
    display(js)

**Step 5: Convert Audio to Text**

In [95]:
def recognize_speech(audio_file):
    recognizer = sr.Recognizer()

    with sr.AudioFile(audio_file) as source:
        audio = recognizer.record(source)

    try:
        text = recognizer.recognize_google(audio)

        print("You:", text)

        return text.lower()

    except:
        return ""

**Step 6: Weather using Open-Meteo**

In [96]:
import requests

def get_weather(city):

    # Step 1: Convert city name to coordinates
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"

    geo_response = requests.get(geo_url).json()

    if "results" not in geo_response:
        return "City not found"

    latitude = geo_response["results"][0]["latitude"]
    longitude = geo_response["results"][0]["longitude"]

    # Step 2: Get weather
    weather_url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}"
        f"&current=temperature_2m,weather_code"
    )

    weather_response = requests.get(weather_url).json()

    temperature = weather_response["current"]["temperature_2m"]

    return f"The current temperature in {city} is {temperature} degree Celsius."

print(get_weather("Kolkata"))

The current temperature in Kolkata is 27.8 degree Celsius.


**Step 7: News using BBC RSS Feed**

In [103]:
import feedparser

def get_news():

    feed = feedparser.parse(
        "https://feeds.bbci.co.uk/news/rss.xml"
    )

    headlines = []

    for entry in feed.entries[:5]:
        headlines.append(entry.title)

    return headlines

news = get_news()

for item in news:
    print(item)

Lammy says he told JD Vance his Nowak comments were 'wrong'
Hegseth attacks Europe over migration with beach 'invasion' D-Day speech
Delays to defence plan undermine UK credibility, MPs say
Steve Rosenberg: Russia's economic forum overshadowed by drone attacks on St Petersburg
Cosmeticorexia: How girls are falling down a skincare rabbit hole


**Step 8: Reminder System**

In [104]:
def set_reminder(message, seconds):
    speak(f"Reminder set for {seconds} seconds")

    time.sleep(seconds)

    speak(f"Reminder. {message}")

set_reminder("Drink water", 10)

Assistant: Reminder set for 10 seconds
Assistant: Reminder. Drink water


**Step 9: Current Time Function**

In [108]:
import datetime

from datetime import datetime
from zoneinfo import ZoneInfo

def get_time():
    ist_time = datetime.now(ZoneInfo("Asia/Kolkata"))
    return ist_time.strftime("%I:%M %p IST")

**Step 10: Command Processing Engine**

In [130]:
def process_command(command):

    command = command.lower().strip()

    # Greetings
    if any(word in command for word in ["hello", "hi", "hey"]):
        speak("Hello, how can I help you?")

    # Time
    elif "time" in command:
        speak(f"The current time is {get_time()}")

    # Weather / Temperature

    elif any(keyword in command for keyword in [
        "weather",
        "temperature",
        "climate",
        "forecast"
    ]):

        try:

            city = command

            # Remove common phrases
            phrases = [
                "weather",
                "temperature",
                "climate",
                "forecast",
                "what is the",
                "tell me the",
                "at",
                "in",
                "of",
                "for"
            ]

            for phrase in phrases:
                city = city.replace(phrase, "")

            city = city.strip()

            if not city:
                speak("Please specify a city name.")
                return

            weather = get_weather(city)

            speak(weather)

        except Exception:
            speak("Unable to fetch weather information.")

    # News

    elif any(keyword in command for keyword in [
        "news",
        "headline",
        "headlines",
        "latest news",
        "current news",
        "today news",
        "today's news",
        "top news",
        "top headlines",
        "breaking news",
        "news update"
    ]):

        try:

            headlines = get_news()

            if not headlines:
                speak("Sorry, I could not find any news at the moment.")
                return

            print("Assistant: Here are the latest headlines.")

            speech_text = "Here are the latest headlines. "

            for i, news in enumerate(headlines[:5], start=1):

                print(f"Assistant: News {i}: {news}")

                speech_text += f"Headline {i}. {news}. "

            speak(speech_text, print_text=False)

        except Exception:

            speak(
                "Sorry, I am unable to fetch the news right now."
            )

    # Reminder
    elif any(keyword in command for keyword in [
        "reminder",
        "remind me",
        "set reminder",
        "set a reminder"
    ]):

        try:

            words = command.split()

            # Find first number in the command
            seconds = None

            for word in words:
                if word.isdigit():
                    seconds = int(word)
                    break

            if seconds is None:
                speak(
                    "Please specify the reminder time in seconds."
                )
                return

            # Remove common reminder phrases
            message = command

            phrases = [
                "reminder",
                "remind me",
                "set reminder",
                "set a reminder",
                "for",
                "in",
                "seconds",
                "second",
                "to"
            ]

            for phrase in phrases:
                message = message.replace(phrase, "")

            # Remove the number from the message
            message = message.replace(str(seconds), "")

            message = message.strip()

            if not message:
                message = "Reminder"

            set_reminder(message, seconds)

        except Exception:

            speak(
                "Please say something like: remind me in 10 seconds to drink water."
            )

    # Exit / Goodbye

    elif any(keyword in command for keyword in [
        "exit",
        "quit",
        "bye",
        "goodbye",
        "see you",
        "thank you",
        "thanks"
    ]):

        speak(
            "You're welcome. Goodbye and have a great day."
        )

    # Unknown command
    else:

        speak(
            "Sorry, I did not understand that command."
        )

**Step 11: Interactive Assistant Loop**

In [136]:
import ipywidgets as widgets
from IPython.display import display

command_box = widgets.Text(
    placeholder='Type your command and press Enter...',
    description='You:',
    layout=widgets.Layout(width='600px')
)

send_button = widgets.Button(
    description='Send',
    button_style='primary'
)

output_area = widgets.Output(
    layout={
        'border': '1px solid lightgray',
        'height': '300px',
        'overflow_y': 'auto'
    }
)


def execute_command(command):

    with output_area:

        print(f"You: {command}")

        if command in ["exit", "quit", "bye"]:

            print("Assistant: Goodbye. Have a great day.")

            speak("Goodbye. Have a great day.")

            command_box.disabled = True
            send_button.disabled = True

            return

        process_command(command)

        print("-" * 60)


def submit_command(_=None):

    command = command_box.value.strip().lower()

    if command:

        execute_command(command)

    command_box.value = ""


# Send button
send_button.on_click(submit_command)

# Enter key
command_box.on_submit(
    lambda sender: submit_command()
)

display(
    widgets.VBox([
        widgets.HTML("<h3>Voice Activated Personal Assistant</h3>"),
        command_box,
        send_button,
        output_area
    ])
)
